# The mobile base

Reachy 2 is mounted on a mobile base!

## Initialize your robot

First connect to your robot:

In [ ]:
from reachy2_sdk import ReachySDK
import time

reachy = ReachySDK(host='localhost')  # Replace with the actual IP

Let's check what contains the mobile base part:

In [ ]:
reachy.mobile_base

## Odometry

## Move around with gotos

The goto commands and based commands described below follow all the rules you saw in the [goto introduction tutorial](2_goto_introduction.ipynb).

### Goto and odometry

The goto function is used to place the mobile_base at a relative position and orientation to its odometry, set when the robot is switched on. To be sure, you can reset the odometry before calling the function. 

In [ ]:
reachy.mobile_base.reset_odometry()

In [ ]:
reachy.mobile_base.odometry

Let's turn on the mobile to be able to move is around

In [ ]:
reachy.mobile_base.turn_on()

The robot is currently positionned at x=0, y=0, theta=0.  
If you want to move forward again the robot, you need to increase the x value (value is in meters):

In [ ]:
# Move 20 cm forward
a = reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

Now, request `goto(0, 0, 0)`. The robot will return to its previous position:

In [ ]:
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0)

All the positions are relative to the fixed odometry coordinate system of the mobile base, set at the start of the robot of after a `reset_odometry()` asked by the user.

So if you do:

In [ ]:
# Move 30cm forward, to reach x=30cm
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0)

# Go back by 10cm, to reach x=20cm
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

The mobile is first going to the position x=30cm in the odometry frame. We then ask for it to get to the position x=20cm in this same frame, so the mobile base is going backward by 10cm to reach its new target.  

Let's do the same by resetting the odometry between the two commands:

In [ ]:
# Move 30cm forward, to reach x=30cm in the current odometry frame
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, wait=True)
time.sleep(0.5)
print(f"x position before odometry reset: {round(reachy.mobile_base.odometry['x'], 2)}")

# Reset odometry
reachy.mobile_base.reset_odometry()
time.sleep(0.5)
print(f"x position after odometry reset: {round(reachy.mobile_base.odometry['x'], 2)}")

# Move 20cm forward, to reach x=20cm in the new current odometry frame
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

As we reset the odometry between the two commands, the mobile base odometry position is reset to x=0cm before the second command. It will then reach x=20cm in the new frame, so move forward by 20cm.

We recommend experimenting with this concept to get familiar.

### Goto tolerances

Unlike the arms and head whose movements duration is based on a duration argument, a mobile base goto duration is based on the tolerances and timeout arguments you can choose.  

You can modify two different tolerances:
- the **distance_tolerance**: defines the maximum allowed distance to the (x, y) position in meters to consider the position as reached
- the **angle_tolerance**: defines the maximum allowed angle distance to the theta rotation to consider the position as reached (units based on the degrees argument, in degrees by default)

Note that the mobile base does not have a precision to the nearest millimeter, so giving a very small tolerance may reach in a movement that will timeout as the tolerance would never be observed.  

> Default distance_tolerance is **0.05 meter**  
> Default angle_tolerance is **5 degrees**

In [ ]:
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0)
time.sleep(3)
print(f"Playing GoToId after 3 seconds: {reachy.mobile_base.get_goto_playing().id}")

After 3 seconds, `get_goto_playing()` returns an GoToId equals to -1, which means the goto is over. The requested position is considered as reached. Let's then check the robot current odometry:

In [ ]:
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

The values may not be exactly the requested ones, but all are within the requested tolerances.  

We can now try to reduce those tolerances to see what happens:

In [ ]:
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, wait=True)
reduced_tol_goto = reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, distance_tolerance=0.0)
print(f"Reduced tolerance GoToId: {reduced_tol_goto.id}")
time.sleep(3)
print(f"Playing GoToId after 3 seconds: {reachy.mobile_base.get_goto_playing().id}")
time.sleep(7)
print(f"Playing GoToId after 10 seconds: {reachy.mobile_base.get_goto_playing().id}")

After 10 seconds, the movement is still not finished. Let's have a look to the odometry:

In [ ]:
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

The distance between the requested position and the current one is higher than the given tolerance. Therefore the target is not considered as reached, so the mobile_base is stuck in the goto, but the current PID of the mobile does not enable it to move from its current position to reach this very close target. The goto will then wait for the timeout.  

### Goto timeout

Let's stop our previous goto and give a custom timeout value:

In [ ]:
# Cancel the stuck goto
reachy.cancel_goto_by_id(reduced_tol_goto)

# Send a new goto with timeout reduced to 5 seconds
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, distance_tolerance=0.001, wait=True)
#reduced_timeout_goto = reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, distance_tolerance=0.001, timeout=5)
print(f"Reduced tolerance GoToId: {reduced_timeout_goto.id}")
time.sleep(3)
print(f"Playing GoToId after 3 seconds: {reachy.mobile_base.get_goto_playing().id}")
time.sleep(7)
print(f"Playing GoToId after 10 seconds: {reachy.mobile_base.get_goto_playing().id}")


If we check the odometry now:

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 5)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

The distance to the tolerance is still not good, but the movement has stopped after the given timeout of 5 seconds.

> Default timeout is **100 seconds**

You can also use the timeout to interrupt a goto after a certain amount of time in a given direction. For example:

In [ ]:
# Go back to base position
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, wait=True)

# Start movement torwards x=1m
timeout_goto = reachy.mobile_base.goto(x=1, y=0.0, theta=0.0, timeout=2)
tic = time.time()
while not reachy.is_goto_finished(timeout_goto):
    time.sleep(0.05)
print(f"goto interrupted after {round(time.time() - tic, 2)} seconds")

Be careful that the mobile does not precisely follow the expected trajectory to reach the target position.

### Relative moves

You can also decide to assign movements to the robot based on its current position and not on its odometry. 

Two methods are available to give relative orders:
- **`translate_by()`**: to give translations orders. 
- **`rotate_by()`**: to give rotations orders  

These methods work like all gotos, and return a GoToId.  

The translation or rotation is computed based on the current position if no goto is playing, or on the position required for the last queued or playing goto in case gotos are not finished.

Let's try some examples to better understand how it works.

#### translate_by()

Send the mobile base back the odometry frame origin first, and reset it:

In [ ]:
a = reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, wait=True, timeout=10)
time.sleep(0.5)
reachy.mobile_base.reset_odometry()

Now, we are going to compare a `goto()` to a `translate_by()` command.  

If we check the odometry, we will see the mobile_base at the origin of the current frame (that we have just reset):

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

You can move forward to the position x=20cm by asking a translation of 20cm on the x-axis:

In [ ]:
reachy.mobile_base.translate_by(x = 0.2, y = 0.0)

Because you started from the origin, the result is the same as asking a `goto(x=0.2, y=0, theta=0)`.
If we send this goto:

In [ ]:
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0, wait=True, timeout=10)

The mobile does not move, because we are already at this position. We can check this using the odometry:

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

But if we ask a new translation of 20cm, the mobile base will go forward:

In [ ]:
reachy.mobile_base.translate_by(x = 0.2, y = 0.0, wait=True, timeout=5)
time.sleep(0.5)

# Read odometry
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

#### rotate_by()

The `rotate_by()` method works quite the same way as the `translate_by()`, unlike it is for rotations.  
For example, you can go back to the initial position then rotate the mobile base. 

In [ ]:
# Go back to the initial position
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, wait=True)

# Rotation to be at 90 degrees in the frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=90.0, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

Now, the mobile base can be rotated 90° from its current position, allowing to get a odometry with a theta = 0°. 

In [ ]:
reachy.mobile_base.rotate_by(theta=-90.0, wait=True)
time.sleep(0.5)

# Check the odometry
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

#### Choose the right method!

Be careful with the method you use. For example, those two sequences have completely different results:

In [ ]:
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0, y=0.4, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0, y=0, theta=90.0, timeout=10, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

In [ ]:
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.translate_by(x=0.2, y=0.0, timeout=10)
reachy.mobile_base.translate_by(x=0, y=0.4, timeout=10)
a=reachy.mobile_base.rotate_by(theta=90.0, timeout=10, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

## Modes

Three modes are possible to control the mobile base:
- **goto** : move the mobile base to a target point in space -> use a *goto function* to get in this mode
- **free wheel**: unlock the wheel so Reachy can be manually moved around easily -> *turn_off() method* will set this mode
- **brake**: stop the movement and lock the wheels -> *turn_on() method* will set this mode

The speed of the movement can be defined using this command : *this will assign speed to the robot for 200ms*

In [ ]:
reachy.mobile_base.set_goal_speed(x=1.0, y=0.0, theta=0)
tic=time.time()
while time.time()-tic < 2:
    reachy.mobile_base.send_speed_command()
    time.sleep(0.01)

### Free wheel

In [ ]:
reachy.mobile_base.turn_off()